In [1]:
import sys
sys.path.append('../../../../')

In [2]:
from CADETProcess.optimization import OptimizationProblem
optimization_problem = OptimizationProblem('evaluation_object_demo')

from examples.batch_elution.process import process

[INFO 08-12 16:18:34] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.52


In [3]:
optimization_problem.add_evaluation_object(process)

In [4]:
optimization_problem.add_variable(
    'var_0',
    evaluation_objects=[process],
    parameter_path='flow_sheet.column.total_porosity',
    lb=0, ub=1,
)

RangedParameter(name='var_0', parameter_type=float, lb=0, ub=1)

In [5]:
optimization_problem = OptimizationProblem('evaluation_object_demo')
optimization_problem.add_evaluation_object(process)

In [6]:
optimization_problem.add_variable('flow_sheet.column.total_porosity', lb=0, ub=1)

RangedParameter(name='flow_sheet.column.total_porosity', parameter_type=float, lb=0, ub=1)

In [7]:
optimization_problem = OptimizationProblem('evaluation_object_demo_multi')

import copy

process_a = copy.deepcopy(process)
process_a.name = 'process_a'
process_b = copy.deepcopy(process)
process_b.name = 'process_b'

In [8]:
optimization_problem.add_evaluation_object(process_a)
optimization_problem.add_evaluation_object(process_b)
optimization_problem.add_variable('flow_sheet.column.total_porosity')
optimization_problem.add_variable('flow_sheet.column.length', evaluation_objects=[process_a])

RangedParameter(name='flow_sheet.column.length', parameter_type=float, lb=-inf, ub=inf)

In [9]:
optimization_problem = OptimizationProblem('evaluator_demo')
optimization_problem.add_variable('x')

RangedParameter(name='x', parameter_type=float, lb=-inf, ub=inf)

In [10]:
def evaluator(x):
    return x**2

optimization_problem.add_evaluator(evaluator)

In [11]:
def objective(result):
    return result + 1

optimization_problem.add_objective(objective, requires=[evaluator])

In [12]:
optimization_problem.evaluate_objectives(2)

array([[5.]])

In [13]:
optimization_problem = OptimizationProblem('shared_evaluator_demo')
optimization_problem.add_variable('x')

RangedParameter(name='x', parameter_type=float, lb=-inf, ub=inf)

In [14]:
def simulate(x):
    print(f"simulate called with {x}")
    return {"yield": x * 0.8, "pressure": x * 1.2}

optimization_problem.add_evaluator(simulate)

def yield_objective(sim_result):
    return sim_result["yield"]

def pressure_constraint(sim_result):
    return sim_result["pressure"]

optimization_problem.add_objective(yield_objective, requires=[simulate])
optimization_problem.add_nonlinear_constraint(pressure_constraint, requires=[simulate])

In [15]:
print("objectives:", optimization_problem.evaluate_objectives(5))
print("constraints:", optimization_problem.evaluate_nonlinear_constraints(5))

simulate called with [5.]
objectives: [[4.]]
constraints: [[6.]]


In [16]:
optimization_problem = OptimizationProblem('chain_demo')
optimization_problem.add_variable('x')

RangedParameter(name='x', parameter_type=float, lb=-inf, ub=inf)

In [17]:
def simulate(x):
    return {"chromatogram": x * 2}

def fractionate(sim_result):
    return {"yield": sim_result["chromatogram"] * 0.9}

optimization_problem.add_evaluator(simulate)
optimization_problem.add_evaluator(fractionate)

def compute_yield(frac_result):
    return frac_result["yield"]

optimization_problem.add_objective(compute_yield, requires=[simulate, fractionate])
optimization_problem.evaluate_objectives(3)

array([[5.4]])

In [18]:
optimization_problem = OptimizationProblem('bypass_cache_demo')
optimization_problem.add_variable('x')

calls = []

def flaky_simulate(x):
    calls.append(x)
    return x * 2

optimization_problem.add_evaluator(flaky_simulate)

def objective(result):
    return result

optimization_problem.add_objective(objective, requires=[flaky_simulate])

In [19]:
optimization_problem.evaluate_objectives(3)
optimization_problem.evaluate_objectives(3)
len(calls)

1

In [20]:
optimization_problem.backend.evaluate({'x': 3}, bypass_cache=True)
len(calls)

2

In [21]:
from dataclasses import dataclass

from CADETProcess.parameter_space import ParameterSpace, RangedParameter
from CADETProcess.evaluation_pipeline import EvaluationPipeline

@dataclass
class Column:
    length: float = 0.5

column = Column()
space = ParameterSpace()
space.add_evaluation_object(column)
space.add_parameter(RangedParameter('length', float, lb=0.1, ub=10.0), path='length')

pipeline = EvaluationPipeline(space)
pipeline.add_evaluator(lambda col: col.length ** 2, output_name='length_squared')
pipeline.evaluate({'length': 3})

{'length_squared': 9.0}